# Refined: Preparando tabelas e visões para consumo
Para essa etapa já posso começar a pensar em um pipeline analítico. Para centralizar os dados farei uso do duckdb e inclusive subir as tabelas trusted como views e gerando um lineage mais rastrável de dados.

In [1]:
import duckdb
from pathlib import Path

path_db = Path(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\duckdb\ml_ettj26.duckdb")
con = duckdb.connect(path_db)
con.sql("SHOW TABLES")


┌────────────────────────────────────┐
│                name                │
│              varchar               │
├────────────────────────────────────┤
│ vw_ref_br_calendar                 │
│ vw_trusted_b3_forwards_di1_lineage │
│ vw_trusted_b3_forwards_di1_quotes  │
│ vw_trusted_b3_forwards_metadata    │
│ vw_trusted_b3_swaps_dixpre_quotes  │
│ vw_trusted_b3_swaps_lineage        │
│ vw_trusted_b3_swaps_metadata       │
│ vw_trusted_bcb_demab_instruments   │
│ vw_trusted_bcb_demab_quotes        │
│ vw_trusted_bcb_sgs_metadata        │
│ vw_trusted_bcb_sgs_points          │
│ vw_trusted_calendar_br             │
│ vw_trusted_holidays_br             │
└────────────────────────────────────┘
               13 rows              

## Calendar: dimensão

Começando pelo mais simples, refinar as tabelas de calendário, sendo elas um dos principais insumos para toda a aplicação, tanto produtos quanto as conveções e curvas. Para todos os efeitos temos duas conveções principais: Dias Corridos e Dias Úteis, sendo aqui a primeira decisão a se tomar para fins de velocidade e praticidade: Ao invés de realizar operações com datetime e cálculos com fins de semana e feriado toda vez que for precificar um produto ou calcular um fator, criei uma formatação de índice, sendo assim otimizando a operação com valores inteiros e qualquer distância entre dadtas se torna uma subtração ao invés de uma busca.

Outro tipo de coluna interessante além das data e índices, seriam flags indicando se é dia útil, feriado ou fim de semana, facilitando filtros com booleanos. Já pensando em views, agregações e análises, também colunas que evitem recálculos extensivos com ano, mês dia do mês e dia da semana serão extremamente úteis agilizando trabalho quando construindo dashboards e definindo regras de negócio, tomando por exemplo produtos que seguem a IMM por exemplo que vencem sempre no dia útil mais próximo do 20 de meses específicos, ou produtos que vencem sempre na segunda quarta do mês.

outras colunas de suporte como lineage da fonte que facilitam o processo de governança da informação e nome de feriado que é mais usado para o cálculo interno das outras colunas também será mantido.

In [2]:
import pandas as pd
holidays = pd.read_parquet(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\calendars\02_trusted\ref\anbima_holidays.parquet")
calendar_bd = pd.read_parquet(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\calendars\02_trusted\ref\calendar_bd_index.parquet")

print(holidays.info())
holidays.head()

print(calendar_bd.info())
calendar_bd.head()

<class 'pandas.DataFrame'>
RangeIndex: 1263 entries, 0 to 1262
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   cal_id            1263 non-null   str                
 1   date              1263 non-null   datetime64[ms, UTC]
 2   holiday_name      1263 non-null   str                
 3   weekday           1263 non-null   int32              
 4   ingestion_ts_utc  1263 non-null   str                
 5   source_file_hash  1263 non-null   str                
 6   pipeline_run_id   1263 non-null   str                
dtypes: datetime64[ms, UTC](1), int32(1), str(5)
memory usage: 214.4 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 36159 entries, 0 to 36158
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   cal_id            36159 non-null  str                
 1   date              

,cal_id,date,weekday,is_business_day,bd_index,holiday_name,ingestion_ts_utc,source_file_hash,pipeline_run_id
0,BR_ANBIMA,2001-01-01 00:00:00+00:00,0,False,0,Confraternização Universal,2026-02-23T01:13:13+00:00,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...,local
1,BR_ANBIMA,2001-01-02 00:00:00+00:00,1,True,1,NaN,2026-02-23T01:13:13+00:00,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...,local
2,BR_ANBIMA,2001-01-03 00:00:00+00:00,2,True,2,NaN,2026-02-23T01:13:13+00:00,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...,local
3,BR_ANBIMA,2001-01-04 00:00:00+00:00,3,True,3,NaN,2026-02-23T01:13:13+00:00,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...,local
4,BR_ANBIMA,2001-01-05 00:00:00+00:00,4,True,4,NaN,2026-02-23T01:13:13+00:00,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...,local


A partir disso e tendo os dois principais arquivos de fonte (lista de feriados ANBIMA e uma trusted de Calendário) podemos ver que o arquivo de calendário é a trusted já num formato semi pronto para consumo, precisando de apenas alguns tratamentos e cáculos para estar pronto para consumo

In [24]:
df = pd.read_parquet(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\calendars\03_refined\Calendar_BR_dim.parquet")
df

,calendar_id,date,year,month,day,weekday,is_weekend,is_holiday,is_business_day,bd_index,holiday_name,source_file_hash
0,BR_ANBIMA,2001-01-01 00:00:00+00:00,2001,1,1,0,False,True,False,0,Confraternização Universal,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
1,BR_ANBIMA,2001-01-02 00:00:00+00:00,2001,1,2,1,False,False,True,1,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
2,BR_ANBIMA,2001-01-03 00:00:00+00:00,2001,1,3,2,False,False,True,2,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
3,BR_ANBIMA,2001-01-04 00:00:00+00:00,2001,1,4,3,False,False,True,3,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
4,BR_ANBIMA,2001-01-05 00:00:00+00:00,2001,1,5,4,False,False,True,4,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
...,...,...,...,...,...,...,...,...,...,...,...,...
36154,BR_ANBIMA,2099-12-27 00:00:00+00:00,2099,12,27,6,True,False,False,24812,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
36155,BR_ANBIMA,2099-12-28 00:00:00+00:00,2099,12,28,0,False,False,True,24813,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
36156,BR_ANBIMA,2099-12-29 00:00:00+00:00,2099,12,29,1,False,False,True,24814,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...
36157,BR_ANBIMA,2099-12-30 00:00:00+00:00,2099,12,30,2,False,False,True,24815,NaN,bff506cbcab013530f5b522ad8b0b2cec0c967dce9b245...


In [2]:
import duckdb as db
from pathlib import Path

con = db.connect(Path(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\duckdb\ml_ettj26.duckdb"))

con.execute("""
            CREATE OR REPLACE VIEW vw_ref_br_calendar AS
            SELECT *
            FROM 'C:/Users/Dell/OneDrive/Documentos/GitHub/ML-ETTJ26/data/calendars/03_refined/Calendar_BR_dim.parquet'
""")

con.sql("SHOW TABLES")

┌────────────────────────────────────┐
│                name                │
│              varchar               │
├────────────────────────────────────┤
│ vw_ref_br_calendar                 │
│ vw_trusted_b3_forwards_di1_lineage │
│ vw_trusted_b3_forwards_di1_quotes  │
│ vw_trusted_b3_forwards_metadata    │
│ vw_trusted_b3_swaps_dixpre_quotes  │
│ vw_trusted_b3_swaps_lineage        │
│ vw_trusted_b3_swaps_metadata       │
│ vw_trusted_bcb_demab_instruments   │
│ vw_trusted_bcb_demab_quotes        │
│ vw_trusted_bcb_sgs_metadata        │
│ vw_trusted_bcb_sgs_points          │
│ vw_trusted_calendar_br             │
│ vw_trusted_holidays_br             │
└────────────────────────────────────┘
               13 rows              

## BCB data: Dados Históricos do BACEN

Partindo para proxima etapa temos dois objetos para estruturar:
- Dados históricos da SELIC provenientes da SGS
- Dados históricos dos títulos públicos negociados pelo DEMAB



### SGS

Os dados históricos da taxa SELIC e índice de inflação IPCA são os mais simples de desenvolver as views por já estarem bem estruturados, sendo nescessário apenas um merge e filtro.

In [26]:
df_sgs_meta = pd.read_parquet(Path(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\02_trusted\bcb\sgs\series_meta.parquet"))
df_sgs_points = pd.read_parquet(Path(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\02_trusted\bcb\sgs\points.parquet"))

print(df_sgs_meta.info())
print(df_sgs_points.info())
df_sgs_meta

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   series_id    2 non-null      int64
 1   series_name  2 non-null      str  
 2   frequency    2 non-null      str  
 3   unit         2 non-null      str  
 4   source       2 non-null      str  
dtypes: int64(1), str(4)
memory usage: 244.0 bytes
None
<class 'pandas.DataFrame'>
RangeIndex: 9852 entries, 0 to 9851
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   series_id         9852 non-null   int64  
 1   ref_date          9852 non-null   object 
 2   value             9852 non-null   float64
 3   record_hash       9852 non-null   str    
 4   raw_file          9852 non-null   str    
 5   raw_hash          9852 non-null   str    
 6   ingestion_ts_utc  9852 non-null   str    
dtypes: float64(1), int64(1), object(1), str(4)
memory us

,series_id,series_name,frequency,unit,source
0,432,SELIC,D,% a.a.,BCB_SGS
1,433,IPCA,M,%,BCB_SGS


In [ ]:
import duckdb as db

path_sgs_meta = "C:/Users/Dell/OneDrive/Documentos/GitHub/ML-ETTJ26/data/02_trusted/bcb/sgs/series_meta.parquet"
path_sgs_points = "C:/Users/Dell/OneDrive/Documentos/GitHub/ML-ETTJ26/data/02_trusted/bcb/sgs/points.parquet"

path_db = Path(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\duckdb\ettj26.duckdb")
con = db.connect(path_db)

df_selic = con.execute(f""" 
    WITH
    
    meta AS (
        SELECT *
        FROM read_parquet('{path_sgs_meta}')
        WHERE series_id = 432
    ),

    points AS (
        SELECT series_id, ref_date, value, raw_hash
        FROM read_parquet('{path_sgs_points}')
        WHERE series_id = 432
    )

    SELECT p.ref_date AS date,
           m.series_id,
           m.series_name,
           p.value,
           m.unit,
           m.frequency,
           m.source,
           p.raw_hash
    FROM meta AS m
    JOIN points AS p
    USING (series_id)
    ORDER BY ref_date
""").df()

df_ipca = con.execute(f"""
    WITH
    
    meta AS (
        SELECT *
        FROM read_parquet('{path_sgs_meta}')
        WHERE series_id = 432
    ),

    points AS (
        SELECT series_id, ref_date, value, raw_hash
        FROM read_parquet('{path_sgs_points}')
        WHERE series_id = 432
    )

    SELECT p.ref_date AS date,
           m.series_id,
           m.series_name,
           p.value,
           m.unit,
           m.frequency,
           m.source,
           p.raw_hash
    FROM meta AS m
    JOIN points AS p
    USING (series_id)
    ORDER BY ref_date 
""").df()

df_selic

,date,series_id,series_name,value,unit,frequency,source,raw_hash
0,2000-01-01,432,SELIC,19.0,% a.a.,D,BCB_SGS,023c3e8e15126a84ae1b8a9e00a325719c03eb7c8f8bbb...
1,2000-01-02,432,SELIC,19.0,% a.a.,D,BCB_SGS,023c3e8e15126a84ae1b8a9e00a325719c03eb7c8f8bbb...
2,2000-01-03,432,SELIC,19.0,% a.a.,D,BCB_SGS,023c3e8e15126a84ae1b8a9e00a325719c03eb7c8f8bbb...
3,2000-01-04,432,SELIC,19.0,% a.a.,D,BCB_SGS,023c3e8e15126a84ae1b8a9e00a325719c03eb7c8f8bbb...
4,2000-01-05,432,SELIC,19.0,% a.a.,D,BCB_SGS,023c3e8e15126a84ae1b8a9e00a325719c03eb7c8f8bbb...
...,...,...,...,...,...,...,...,...
9534,2026-02-07,432,SELIC,15.0,% a.a.,D,BCB_SGS,480c424330ea60ca1d9e880bb515e8be38bb0b61f0c3d6...
9535,2026-02-08,432,SELIC,15.0,% a.a.,D,BCB_SGS,480c424330ea60ca1d9e880bb515e8be38bb0b61f0c3d6...
9536,2026-02-09,432,SELIC,15.0,% a.a.,D,BCB_SGS,480c424330ea60ca1d9e880bb515e8be38bb0b61f0c3d6...
9537,2026-02-10,432,SELIC,15.0,% a.a.,D,BCB_SGS,480c424330ea60ca1d9e880bb515e8be38bb0b61f0c3d6...


In [ ]:
import duckdb as db
from pathlib import Path

con = db.connect(Path(r"C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\duckdb\ml_ettj26.duckdb"))

con.sql("SHOW TABLES")

┌────────────────────────────────────┐
│                name                │
│              varchar               │
├────────────────────────────────────┤
│ vw_trusted_b3_forwards_di1_lineage │
│ vw_trusted_b3_forwards_di1_quotes  │
│ vw_trusted_b3_forwards_metadata    │
│ vw_trusted_b3_swaps_dixpre_quotes  │
│ vw_trusted_b3_swaps_lineage        │
│ vw_trusted_b3_swaps_metadata       │
│ vw_trusted_bcb_demab_instruments   │
│ vw_trusted_bcb_demab_quotes        │
│ vw_trusted_bcb_sgs_metadata        │
│ vw_trusted_bcb_sgs_points          │
│ vw_trusted_calendar_br             │
│ vw_trusted_holidays_br             │
└────────────────────────────────────┘
               12 rows              

### DEMAB
Já para os dados da DEMAB temos a complexidade de muitos dados diários, porém o duckdb consegue lidar muito bem com isso sem a nescessidade de listar todos os arquivos do diretório ou outras técnicas de mais baixo nível. 

In [77]:
import os
path_data = "C:/Users/Dell/OneDrive/Documentos/GitHub/ML-ETTJ26/data"
path_demab = os.path.join(path_data, "02_trusted", "bcb", "demab")
path_demab_quotes = os.path.join(path_demab,  "quotes_daily", "*.parquet" )
path_demab_masters = os.path.join(path_demab,  "instruments.parquet" )

df_demab_ref = con.execute(f"""
    SELECT *
    FROM read_parquet('{path_demab_quotes}') AS quotes
    JOIN read_parquet('{path_demab_masters}') AS masters
    USING (isin)
    """).df()
df_demab_ref.info()

<class 'pandas.DataFrame'>
RangeIndex: 197738 entries, 0 to 197737
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   trade_date        197738 non-null  datetime64[us]
 1   isin              197738 non-null  str           
 2   pu_min            187169 non-null  float64       
 3   pu_med            187169 non-null  float64       
 4   pu_max            187169 non-null  float64       
 5   pu_lastro         197738 non-null  float64       
 6   valor_par         197522 non-null  float64       
 7   taxa_min          102480 non-null  float64       
 8   taxa_med          102480 non-null  float64       
 9   taxa_max          102478 non-null  float64       
 10  raw_zip_file      197738 non-null  str           
 11  raw_zip_hash      197738 non-null  str           
 12  inner_file        197738 non-null  str           
 13  record_hash       197738 non-null  str           
 14  ingestion_ts_ut

In [91]:
df_demab_ref.head()

,trade_date,isin,pu_min,pu_med,pu_max,pu_lastro,valor_par,taxa_min,taxa_med,taxa_max,raw_zip_file,raw_zip_hash,inner_file,record_hash,ingestion_ts_utc,sigla,emissao_date,vencimento_date,source
0,2007-03-01,BRSTNCLF17X6,3021.249086,3021.249107,3021.250531,3021.057273,3021.225061,-0.0152,-0.0143,-0.0143,NegE200703.ZIP,9cd8b7fcc3f523fbf1797be8e21b2fcf70491922b33836...,NegE200703.CSV,055f8494d052acc879a5bf3f35ce113fd11325847deb60...,2026-02-17T19:50:51+00:00,LFT,2002-09-19,2007-03-21,BCB_DEMAB
1,2007-03-01,BRSTNCLF1808,3020.769627,3021.335778,3021.356274,3020.314460,3021.225061,-0.0144,-0.0122,0.0500,NegE200703.ZIP,9cd8b7fcc3f523fbf1797be8e21b2fcf70491922b33836...,NegE200703.CSV,439b162adc7ab3103e370ad020144de4d0480457836dde...,2026-02-17T19:50:51+00:00,LFT,2002-09-19,2007-06-20,BCB_DEMAB
2,2007-03-01,BRSTNCLF1832,3021.408353,3021.411162,3021.411597,3019.547876,3021.225061,-0.0111,-0.0111,-0.0109,NegE200703.ZIP,9cd8b7fcc3f523fbf1797be8e21b2fcf70491922b33836...,NegE200703.CSV,696d5ef6c2240bdba9c3ac02d5b95d7d3cf756b1074a5b...,2026-02-17T19:50:51+00:00,LFT,2002-09-19,2007-09-19,BCB_DEMAB
3,2007-03-01,BRSTNCLF1865,3021.225061,3021.448157,3021.501640,3018.805434,3021.225061,-0.0114,-0.0092,0.0000,NegE200703.ZIP,9cd8b7fcc3f523fbf1797be8e21b2fcf70491922b33836...,NegE200703.CSV,79222a27dceddf7ba78430732affafc00116c11b537ad0...,2026-02-17T19:50:51+00:00,LFT,2002-09-20,2007-12-19,BCB_DEMAB
4,2007-03-01,BRSTNCLF1899,3021.498466,3021.518271,3021.518327,3017.760510,3021.225061,-0.0093,-0.0093,-0.0087,NegE200703.ZIP,9cd8b7fcc3f523fbf1797be8e21b2fcf70491922b33836...,NegE200703.CSV,9baa2e6208d80cdff0fa9147bd971d3f2436ffc147c110...,2026-02-17T19:50:51+00:00,LFT,2002-09-20,2008-03-19,BCB_DEMAB


In [62]:
df_demab_quotes = con.execute(f"""
            SELECT *
            FROM '{path_demab_quotes}'
            """).df()

print(df_demab_quotes.info())
df_demab_quotes.head(5)

<class 'pandas.DataFrame'>
RangeIndex: 197738 entries, 0 to 197737
Data columns (total 15 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   trade_date        197738 non-null  datetime64[us]
 1   isin              197738 non-null  str           
 2   pu_min            187169 non-null  float64       
 3   pu_med            187169 non-null  float64       
 4   pu_max            187169 non-null  float64       
 5   pu_lastro         197738 non-null  float64       
 6   valor_par         197522 non-null  float64       
 7   taxa_min          102480 non-null  float64       
 8   taxa_med          102480 non-null  float64       
 9   taxa_max          102478 non-null  float64       
 10  raw_zip_file      197738 non-null  str           
 11  raw_zip_hash      197738 non-null  str           
 12  inner_file        197738 non-null  str           
 13  record_hash       197738 non-null  str           
 14  ingestion_ts_ut

,trade_date,isin,pu_min,pu_med,pu_max,pu_lastro,valor_par,taxa_min,taxa_med,taxa_max,raw_zip_file,raw_zip_hash,inner_file,record_hash,ingestion_ts_utc
0,2007-01-02,BRSTNCLF1741,2963.029180,2963.029180,2963.029180,2962.893408,2963.009787,-0.0150,-0.0150,-0.0150,NegE200701.ZIP,c357e07c5d7074675e559fbbd1dce2fb9fc05c1c869939...,NegE200701.CSV,6fef5bb7caab164bcf40da41a3da62fda98f082bb227ef...,2026-02-17T19:50:51+00:00
1,2007-01-02,BRSTNCLF17W8,2963.069730,2963.076077,2963.076949,2962.650147,2963.009787,-0.0168,-0.0166,-0.0150,NegE200701.ZIP,c357e07c5d7074675e559fbbd1dce2fb9fc05c1c869939...,NegE200701.CSV,753461644ff60997f9d9fec010c3f7f724bf49ab30d70e...,2026-02-17T19:50:51+00:00
2,2007-01-02,BRSTNCLF17X6,2963.009786,2963.090219,2963.091252,2962.375212,2963.009787,-0.0128,-0.0127,0.0000,NegE200701.ZIP,c357e07c5d7074675e559fbbd1dce2fb9fc05c1c869939...,NegE200701.CSV,91cbb15060ee1ef64b0ca470a3b44fe82d89bcaded2b12...,2026-02-17T19:50:51+00:00
3,2007-01-02,BRSTNCLF1808,2963.009786,2963.156685,2963.172108,2961.646827,2963.009787,-0.0119,-0.0108,0.0000,NegE200701.ZIP,c357e07c5d7074675e559fbbd1dce2fb9fc05c1c869939...,NegE200701.CSV,d2c4f77ddfdf6030b5489465e1c22432859ea04edd5c99...,2026-02-17T19:50:51+00:00
4,2007-01-02,BRSTNCLF1832,2963.234096,2963.236206,2963.236466,2960.895134,2963.009787,-0.0107,-0.0107,-0.0106,NegE200701.ZIP,c357e07c5d7074675e559fbbd1dce2fb9fc05c1c869939...,NegE200701.CSV,c8d94f478b12283ab1c8d39730c5265af7b60382bb6011...,2026-02-17T19:50:51+00:00


In [63]:
df_demab_meta = con.execute(f"""
            SELECT *
            FROM '{path_demab_masters}'
            """).df()
print(df_demab_meta.info())
df_demab_meta.tail()

<class 'pandas.DataFrame'>
RangeIndex: 407 entries, 0 to 406
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   isin             407 non-null    str           
 1   sigla            407 non-null    str           
 2   emissao_date     407 non-null    datetime64[us]
 3   vencimento_date  407 non-null    datetime64[us]
 4   source           407 non-null    str           
dtypes: datetime64[us](2), str(3)
memory usage: 26.0 KB
None


,isin,sigla,emissao_date,vencimento_date,source
402,BRSTNCNTF1Y0,NTN-F,2019-10-01,2029-01-01,BCB_DEMAB
403,BRSTNCNTF204,NTN-F,2020-01-10,2031-01-01,BCB_DEMAB
404,BRSTNCNTF212,NTN-F,2022-01-07,2033-01-01,BCB_DEMAB
405,BRSTNCNTF238,NTN-F,2024-01-05,2035-01-01,BCB_DEMAB
406,BRSTNCNTF2K7,NTN-F,2026-01-09,2037-01-01,BCB_DEMAB


In [5]:
df_demab_master = con.sql(f"""
            SELECT *
            FROM '{path_demab_masters}'
            """).df()
print(df_demab_master.info())
df_demab_master.head()

<class 'pandas.DataFrame'>
RangeIndex: 407 entries, 0 to 406
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   isin             407 non-null    str           
 1   sigla            407 non-null    str           
 2   emissao_date     407 non-null    datetime64[us]
 3   vencimento_date  407 non-null    datetime64[us]
 4   source           407 non-null    str           
dtypes: datetime64[us](2), str(3)
memory usage: 26.0 KB
None


,isin,sigla,emissao_date,vencimento_date,source
0,,LTN,2018-07-06,2020-10-01,BCB_DEMAB
1,BRSTNCBTN2H7,BTNBIB,1990-11-15,2013-03-15,BCB_DEMAB
2,BRSTNCBTN2I5,BTNBIB,1990-11-15,2013-09-15,BCB_DEMAB
3,BRSTNCLF0008,LFT,2018-07-06,2024-09-01,BCB_DEMAB
4,BRSTNCLF0735,LFT-A,1999-08-02,2014-08-02,BCB_DEMAB
